# H-mode global confinement $\tau_E$ scaling
Reproduces the basic engineering-variable results of Verdoolaege *et al.* 2021, Nucl. Fusion **61** 076006 (DB5.2.3) from the IMAS-migrated H-mode database.

In [3]:
import os
import numpy as np
import imas

ROOT = os.path.dirname(os.getcwd())
HMODE_DIR = os.path.join(ROOT, "resources", "results", "hmode")

pulse_dirs = sorted(
    os.path.join(HMODE_DIR, d)
    for d in os.listdir(HMODE_DIR)
    if d.startswith("pulse_")
)
N = len(pulse_dirs)
print(f"{N} pulses in {HMODE_DIR}")

14153 pulses in c:\Users\curranf\IDStools\resources\results\hmode


## Scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| $\tau_{E,th}$ (TAUTH) | `summary/global_quantities/tau_energy/value` | s |
| $I_p$ | `summary/global_quantities/ip/value` | A |
| $B_t$ | `summary/global_quantities/b0/value` | T |
| $\bar n_e$ | `summary/line_average/n_e/value` | m^-3 |
| $P_{l,th}$ | `summary/global_quantities/power_loss/value` | W |
| $R_{geo}$ | `summary/global_quantities/r0/value` | m |
| $V$ | `summary/global_quantities/volume/value` | m^3 |
| $a$ | `summary/boundary/minor_radius/value` | m |
| $M_{eff}$ | `summary/volume_average/meff_hydrogenic/value` | AMU |
| $\delta$ | `equilibrium/time_slice(0)/boundary/triangularity` | — |
| TOK | `summary/machine` | — |
| PHASE | `temporary/constant_string0d` (by `identifier.name`) | — |
| SELDB5 | `temporary/constant_integer0d` (by `identifier.name`) | — |

Derived (paper definitions): $\kappa_a = V/(2\pi R_{geo}\,\pi a^2)$, $\epsilon = a/R_{geo}$.

In [4]:
def first_scalar(arr):
    """Return first element of array (NaN if empty/unavailable)."""
    try:
        a = np.asarray(arr, dtype=float)
        return float(a.flat[0]) if a.size else np.nan
    except Exception:
        return np.nan


TAU   = np.full(N, np.nan)          # thermal energy confinement time [s]
IP    = np.full(N, np.nan)          # plasma current [A]
BT    = np.full(N, np.nan)          # vacuum B_t at R0 [T]
NEL   = np.full(N, np.nan)          # line-averaged n_e [m^-3]
PLTH  = np.full(N, np.nan)          # thermal loss power [W]
RGEO  = np.full(N, np.nan)          # geometric major radius [m]
VOL   = np.full(N, np.nan)          # plasma volume [m^3]
AMIN  = np.full(N, np.nan)          # minor radius [m]
MEFF  = np.full(N, np.nan)          # effective hydrogenic mass [AMU] (~PGASA)
DELTA = np.full(N, np.nan)          # triangularity
TOK   = np.empty(N, dtype=object)   # tokamak name
PHASE = np.empty(N, dtype=object)   # discharge phase (ELM type)
SELDB5 = np.full(N, np.nan)         # DB5 standard-selection flag

for i, pulse_dir in enumerate(pulse_dirs):
    uri = f"imas:hdf5?path={pulse_dir};pulse=0"
    if i % 100 == 0:
        print(f"Processing pulse {i}/{N}")
    with imas.DBEntry(uri, "r") as entry:
        s = entry.get("summary", lazy=True)
        TAU[i]   = first_scalar(s.global_quantities.tau_energy.value)
        IP[i]    = first_scalar(s.global_quantities.ip.value)
        BT[i]    = np.abs(first_scalar(s.global_quantities.b0.value))
        NEL[i]   = first_scalar(s.line_average.n_e.value)
        PLTH[i]  = first_scalar(s.global_quantities.power_loss.value)
        RGEO[i]  = first_scalar(s.global_quantities.r0.value)
        VOL[i]   = first_scalar(s.global_quantities.volume.value)
        AMIN[i]  = first_scalar(s.boundary.minor_radius.value)
        MEFF[i]  = first_scalar(s.volume_average.meff_hydrogenic.value)
        TOK[i]   = str(s.machine).strip()

        eq = entry.get("equilibrium", lazy=True)
        DELTA[i] = first_scalar(eq.time_slice[0].boundary.triangularity) if len(eq.time_slice) else np.nan

        tmp = entry.get("temporary", lazy=True)
        PHASE[i]  = str(tmp.constant_string0d[0].value).strip() if len(tmp.constant_string0d) else ""
        SELDB5[i] = first_scalar(tmp.constant_integer0d[1].value) if len(tmp.constant_integer0d) else np.nan

print("Done.")

Processing pulse 0/14153
Processing pulse 100/14153
Processing pulse 200/14153
Processing pulse 300/14153
Processing pulse 400/14153
Processing pulse 500/14153
Processing pulse 600/14153
Processing pulse 700/14153
Processing pulse 800/14153
Processing pulse 900/14153
Processing pulse 1000/14153
Processing pulse 1100/14153
Processing pulse 1200/14153
Processing pulse 1300/14153
Processing pulse 1400/14153
Processing pulse 1500/14153
Processing pulse 1600/14153
Processing pulse 1700/14153
Processing pulse 1800/14153
Processing pulse 1900/14153
Processing pulse 2000/14153
Processing pulse 2100/14153
Processing pulse 2200/14153
Processing pulse 2300/14153
Processing pulse 2400/14153
Processing pulse 2500/14153
Processing pulse 2600/14153
Processing pulse 2700/14153
Processing pulse 2800/14153
Processing pulse 2900/14153
Processing pulse 3000/14153
Processing pulse 3100/14153
Processing pulse 3200/14153
Processing pulse 3300/14153
Processing pulse 3400/14153
Processing pulse 3500/14153
Proc

In [5]:
# Units and derived variables (formula in paper)
tau_s    = TAU                                 # [s]
ip_ma    = np.abs(IP) / 1e6                     # [MA]
Bt_T     = np.abs(BT)                           # [T]
ne_19    = NEL / 1e19                           # [10^19 m^-3]
Ploss_MW = PLTH / 1e6                           # [MW]
kappa_a  = VOL / (2.0 * np.pi * RGEO * np.pi * AMIN**2)   # paper kappa_a
eps      = AMIN / RGEO                          # inverse aspect ratio
one_delta = 1.0 + DELTA                         # 1 + delta

# Subset selection: STD5 standard set, ELMy H-mode
PHASE_str = PHASE.astype(str)
std5 = (SELDB5 == 1)
elmy = np.char.startswith(PHASE_str, "HG") | np.char.startswith(PHASE_str, "HS")

# Check if theres a difference?
print(f"Is std5 == elmy?: {np.array_equal(std5, elmy)}")

if not np.array_equal(std5,elmy):
    print(f"No, they differ in: {np.sum(std5 != elmy)} places")
    

regressors = [tau_s, ip_ma, Bt_T, ne_19, Ploss_MW, RGEO, kappa_a, eps, MEFF]
finite = np.all([np.isfinite(p) for p in regressors], axis=0)
positive = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
sel = elmy & finite & positive

print(f"Total pulses     : {N}")
print(f"SELDB5 == 1      : {np.sum(std5)}")
print(f"ELMy H (HG/HS)   : {np.sum(elmy)}")
print(f"STD5 ELMy + valid: {np.sum(sel)}")
print("\nper machine (STD5 ELMy, valid):")
for tok in np.unique(TOK[sel]):
    print(f"  {tok:10s}: {np.sum((TOK == tok) & sel)}")

Is std5 == elmy?: False
No, they differ in: 2560 places
Total pulses     : 14153
SELDB5 == 1      : 7568
ELMy H (HG/HS)   : 7520
STD5 ELMy + valid: 7427

per machine (STD5 ELMy, valid):
  ASDEX     : 445
  AUG       : 2329
  CMOD      : 46
  COMPASS   : 26
  D3D       : 458
  JET       : 3082
  JFT2M     : 76
  JT60U     : 387
  MAST      : 47
  NSTX      : 201
  PBXM      : 80
  PDX       : 117
  START     : 9
  TCV       : 15
  TDEV      : 10
  TFTR      : 99


## Table 2 — engineering-variable ranges (STD5 ELMy H)
Compare with paper Table 2 (DB5.2.3-STD5 ELMy H). 

In [6]:
cols = {
    "tau_E,th [s]": tau_s[sel],
    "Ip [MA]":      ip_ma[sel],
    "Bt [T]":       Bt_T[sel],
    "ne [1e19]":    ne_19[sel],
    "Pl,th [MW]":   Ploss_MW[sel],
    "Rgeo [m]":     RGEO[sel],
    "1+delta":      one_delta[sel],
    "kappa_a":      kappa_a[sel],
    "eps":          eps[sel],
    "Meff":         MEFF[sel],
}
print(f"{'variable':14s} {'min':>9s} {'max':>9s} {'mean':>9s} {'median':>9s} {'std':>9s}")
for name, x in cols.items():
    print(f"{name:14s} {np.min(x):9.4g} {np.max(x):9.4g} {np.mean(x):9.4g} {np.median(x):9.4g} {np.std(x):9.4g}")

variable             min       max      mean    median       std
tau_E,th [s]    0.002236     1.506    0.1856    0.1257    0.1628
Ip [MA]           0.1524     5.162     1.431     1.028    0.8664
Bt [T]            0.2613     5.821      2.19     2.197    0.7202
ne [1e19]         0.7258      42.9     5.756     5.305     3.189
Pl,th [MW]        0.1464     32.67     8.146      6.85     5.236
Rgeo [m]          0.2804      3.46     2.223      1.69    0.7242
1+delta              nan       nan       nan       nan       nan
kappa_a           0.9308     2.389     1.523     1.561    0.2066
eps               0.1548    0.7831    0.3217    0.3147   0.08067
Meff                   1      3.89     1.928         2    0.2729


## Engineering scaling (OLS in log space)
Power law $\tau_{E,th} = \alpha_0\, I_p^{\alpha_I} B_t^{\alpha_B} \bar n_e^{\alpha_n} P_{l,th}^{\alpha_P} R_{geo}^{\alpha_R} \kappa_a^{\alpha_\kappa} \epsilon^{\alpha_\epsilon} M_{eff}^{\alpha_M}$ (paper eq. 2), fitted by ordinary least squares on $\ln$-transformed data.

In [7]:
# Design matrix in log space (Ip in MA, ne in 1e19, Pl,th in MW — absorbed into intercept)
y = np.log(tau_s[sel])
regressors = {
    "ln Ip":   np.log(ip_ma[sel]),
    "ln Bt":   np.log(Bt_T[sel]),
    "ln ne":   np.log(ne_19[sel]),
    "ln Plth": np.log(Ploss_MW[sel]),
    "ln Rgeo": np.log(RGEO[sel]),
    "ln kapa": np.log(kappa_a[sel]),
    "ln eps":  np.log(eps[sel]),
    "ln Meff": np.log(MEFF[sel]),
}
X = np.column_stack([np.ones_like(y)] + list(regressors.values()))
coef, *_ = np.linalg.lstsq(X, y, rcond=None)

names = ["ln alpha0"] + list(regressors.keys())
# IPB98(y,2) reference exponents (Table 7): aI,aB,an,aP,aR,akappa,aeps,aM
ipb98 = {"ln Ip":0.93, "ln Bt":0.15, "ln ne":0.41, "ln Plth":-0.69,
         "ln Rgeo":1.97, "ln kapa":0.78, "ln eps":0.58, "ln Meff":0.19}

print(f"{'param':10s} {'this OLS':>10s} {'IPB98(y,2)':>12s}")
print(f"{'alpha0':10s} {np.exp(coef[0]):>10.4g} {0.0562:>12.4g}")
for nm, c in zip(names[1:], coef[1:]):
    ref = ipb98.get(nm, np.nan)
    print(f"{nm:10s} {c:>10.3f} {ref:>12.3f}")

resid = y - X @ coef
rmse = np.sqrt(np.mean(resid**2))
r2 = 1.0 - np.sum(resid**2) / np.sum((y - y.mean())**2)
print(f"\nN = {sel.sum()}   RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}")

param        this OLS   IPB98(y,2)
alpha0        0.08646       0.0562
ln Ip           1.115        0.930
ln Bt           0.132        0.150
ln ne           0.216        0.410
ln Plth        -0.696       -0.690
ln Rgeo         1.484        1.970
ln kapa         0.347        0.780
ln eps          0.211        0.580
ln Meff         0.130        0.190

N = 7427   RMSE(log) = 0.212   R^2 = 0.938


## Table 8 — per-machine OLS scalings (all H-modes in STD5)
OLS on log-transformed data. 

In [26]:
from numpy.linalg import lstsq

def ols_fit(y_log, X):
    """OLS in log space; returns (coef, std_err, resid)."""
    coef, _, _, _ = lstsq(X, y_log, rcond=None)
    n, p = X.shape
    resid = y_log - X @ coef
    # unbiased variance of residuals
    sigma2 = np.sum(resid**2) / max(n - p, 1)
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return coef, se, resid

def metrics(resid, y_log, p_free):
    """MdAPE (%), RMSE (log), R^2 — all on log scale"""
    n = len(resid)
    mdape = np.median(np.abs(np.expm1(resid))) * 100   # |exp(r)-1|*100
    rmse  = np.sqrt(np.sum(resid**2) / n)              # no df correction, matches paper
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y_log - y_log.mean())**2)
    r2    = 1.0 - ss_res / ss_tot
    return mdape, rmse, r2

# Paper Table 8 uses "all H-modes" in STD5 (not just ELMy).
# SELDB5 == 1 is the STD5 flag; require all regressors tp be finite & positive.

all_h = (SELDB5 == 1)
preds_all = [tau_s, ip_ma, Bt_T, ne_19, Ploss_MW, RGEO, kappa_a, eps, MEFF, one_delta]
finite_all = np.all([np.isfinite(p) | np.isnan(p) for p in preds_all], axis=0)
# positivity for the mandatory variables only
pos_all = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
mask = all_h & pos_all & np.isfinite(tau_s)

# log-arrays (NaN-safe; we apply device mask before building X)
LOG = {
    "Ip":    np.log(np.where(ip_ma    > 0, ip_ma,    np.nan)),
    "Bt":    np.log(np.where(Bt_T     > 0, Bt_T,     np.nan)),
    "ne":    np.log(np.where(ne_19    > 0, ne_19,    np.nan)),
    "Plth":  np.log(np.where(Ploss_MW > 0, Ploss_MW, np.nan)),
    "Rgeo":  np.log(np.where(RGEO     > 0, RGEO,     np.nan)),
    "kappa":  np.log(np.where(kappa_a  > 0, kappa_a,  np.nan)),
    "eps":   np.log(np.where(eps      > 0, eps,       np.nan)),
    "Meff":  np.log(np.where(MEFF     > 0, MEFF,     np.nan)),
    "1+d":   np.log(np.where(one_delta > 0, one_delta, np.nan)),
}
LOG_TAU = np.log(np.where(tau_s > 0, tau_s, np.nan))


DEVICE_SPECS = {
    # TOK  : (paper_label,  [regressors])
    "ASDEX"  : ("ASDEX",       ["Ip", "Bt", "ne", "Plth", "Meff"]),
    "AUG"    : ("AUG",         ["Ip", "Bt", "ne", "Plth"]),
    "CMOD"   : ("Alcator C-Mod", ["Ip",       "ne", "Plth"]),
    "COMPASS": ("COMPASS-D",   ["Ip", "Bt", "ne", "Plth"]),
    "D3D"    : ("DIII-D",      ["Ip", "Bt", "ne", "Plth", "1+d", "Meff"]),
    "JET"    : ("JET",         ["Ip", "Bt", "ne", "Plth", "kappa", "Meff"]),
    "JFT2M"  : ("JFT-2M",      ["Ip",       "ne", "Plth", "Meff"]),
    "JT60U"  : ("JT-60U",      ["Ip", "Bt", "ne", "Plth"]),
    "MAST"   : ("MAST",        ["Ip",       "ne", "Plth"]),
    "NSTX"   : ("NSTX",        ["Ip", "Bt", "ne", "Plth", "kappa"]),
    "PBXM"   : ("PBX-M",       ["Ip",       "ne", "Plth"]),
    "PDX"    : ("PDX",         ["Ip", "Bt", "ne", "Plth"]),
}

# paper Table 8 exponent columns (in order of predictors listed above)
COL_ORDER = ["Ip", "Bt", "ne", "Plth", "1+d", "kappa", "Meff"]
COL_LABELS = [r"$\alpha_I$", r"$\alpha_B$", r"$\alpha_n$", r"$\alpha_P$", 
              r"$\alpha_{(1+\delta)}$", r"$\kappa$", r"$\alpha_M$"]

header = (f"{'Device':<14s}  {'n_obs':>5s}  "
       + "   ".join(f"{c:>9s}" for c in COL_LABELS)
       + f"  {'MdAPE%':>7s}  {'RMSE':>6s}  {'R^2':>5s}")
print(header)
print("-" * len(header))

results = {}
for tok, (label, pkeys) in DEVICE_SPECS.items():
    m = mask & (TOK == tok)
    # require all regressors for this device to be finite
    log_preds = [LOG[k] for k in pkeys]
    finite_dev = np.all([np.isfinite(lp) for lp in log_preds], axis=0)
    finite_dev &= np.isfinite(LOG_TAU)
    m = m & finite_dev
    n = int(m.sum())
    if n < len(pkeys) + 2:
        print(f"{label:<14s} {n:>5d}  (too few points)")
        continue

    y = LOG_TAU[m]
    cols_X = [np.ones(n)] + [LOG[k][m] for k in pkeys]
    X = np.column_stack(cols_X)
    coef, se, resid = ols_fit(y, X)
    mdape, rmse, r2 = metrics(resid, y, len(pkeys))

    # build coefficient dict keyed by predictor name
    coef_dict = {k: (coef[i+1], se[i+1]) for i, k in enumerate(pkeys)}

    # format row: show value pm standard error (se) for each column in COL_ORDER, blank if not fitted
    row = f"{label:<14s} {n:>5d}  "
    for col in COL_ORDER:
        if col in coef_dict:
            v, s = coef_dict[col]
            row += f"  {v:+5.3f}pm{s:5.3f}"
        else:
            row += f"  {'-------------':>9s}"
    row += f"  {mdape:7.1f}  {rmse:6.2f}  {r2:5.2f}"
    print(row)
    results[tok] = {"n": n, "coef": coef_dict, "mdape": mdape, "rmse": rmse, "r2": r2}


Device          n_obs  $\alpha_I$   $\alpha_B$   $\alpha_n$   $\alpha_P$   $\alpha_{(1+\delta)}$    $\kappa$   $\alpha_M$   MdAPE%    RMSE    R^2
-------------------------------------------------------------------------------------------------------------------------------------------------
ASDEX            575    +0.754pm0.043  +0.220pm0.060  +0.678pm0.032  -0.649pm0.019  -------------  -------------  +0.049pm0.046      9.1    0.13   0.80
AUG             2141    +1.726pm0.029  -0.476pm0.032  -0.164pm0.017  -0.670pm0.012  -------------  -------------  -------------     12.0    0.19   0.77
Alcator C-Mod     82    +1.149pm0.080  -------------  +0.101pm0.091  -0.597pm0.074  -------------  -------------  -------------      7.4    0.10   0.79
COMPASS-D         21    +2.034pm0.542  -0.234pm0.399  +0.711pm0.136  -0.818pm0.232  -------------  -------------  -------------      5.7    0.09   0.85
DIII-D           502    +1.101pm0.047  +0.100pm0.058  +0.094pm0.035  -0.633pm0.021  +0.546pm0.085  -